# SQL to grid: DuckDB over a Parquet file

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/toclocoinc/lattice-grid-notebooks/blob/main/sql-to-grid-duckdb/sql_to_grid_duckdb.ipynb)

Query a Parquet file with DuckDB, see the result profiled in the grid, then **change the query and keep going**: `widget.set_data(...)` swaps in the new result and the grid keeps its column state (widths, sort, and filters on columns that still exist). A big result switches to the large-dataframe mode by itself. No API key.

**Data:** [NYC TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page), yellow-taxi trips for January 2024 (one 50 MB Parquet file, about 2.9 million trips) plus the TLC taxi-zone lookup. Published by the NYC Taxi & Limousine Commission as open data, free to use; fetched from the TLC's public CDN when you run the notebook. The notebook code is MIT-licensed (see the repo `LICENSE`); the data stays under the TLC's terms.

In [1]:
%pip install -q "lattice-grid-jupyter[large]" duckdb

Note: you may need to restart the kernel to use updated packages.


## 1. Point DuckDB at the Parquet file

DuckDB reads the Parquet file in place; a `trips` view joins the zone names so every query below stays short.

In [2]:
import os, urllib.request, duckdb
from lattice_grid_jupyter import LatticeGridWidget

CDN = "https://d37ci6vzurychx.cloudfront.net/"
for name, path in [("yellow_tripdata_2024-01.parquet", "trip-data/"), ("taxi_zone_lookup.csv", "misc/")]:
    if not os.path.exists(name):
        urllib.request.urlretrieve(CDN + path + name, name)

duckdb.sql("""
    CREATE OR REPLACE VIEW trips AS
    SELECT tpep_pickup_datetime AS pickup, pu.Zone AS pickup_zone, dz.Zone AS dropoff_zone,
           trip_distance AS miles, fare_amount AS fare, tip_amount AS tip
    FROM 'yellow_tripdata_2024-01.parquet' t
    JOIN 'taxi_zone_lookup.csv' pu ON t.PULocationID = pu.LocationID
    JOIN 'taxi_zone_lookup.csv' dz ON t.DOLocationID = dz.LocationID
    WHERE trip_distance > 0 AND fare_amount > 0 AND tpep_pickup_datetime >= '2024-01-01'
""")
duckdb.sql("SELECT count(*) AS trips FROM trips")

┌─────────┐
│  trips  │
│  int64  │
├─────────┤
│ 2869700 │
└─────────┘

## 2. A query function, and the first result in a grid

`by_route` is a plain SQL query over the `trips` view: one row per pickup zone, dropoff zone and hour, with its trip count and the average miles, fare and tip (`miles`, `fare`, `tip`). `duckdb.sql(...).df()` returns a pandas DataFrame, which is all the widget needs. `profile=True` adds a **Statistics** tab on the right edge and `histograms=True` puts a distribution in each column header.

The grid starts sorted by trips, busiest route first, and filtered to routes with an average fare over $20. Drag a column wider, re-sort, or change the filter in the grid; the next steps keep what you set.

In [3]:
def by_route(where="true", min_trips=50):
    return duckdb.sql(f"""
        SELECT pickup_zone, dropoff_zone, hour(pickup) AS hour, count(*) AS trips,
               round(avg(miles), 1) AS miles, round(avg(fare), 2) AS fare, round(avg(tip), 2) AS tip
        FROM trips WHERE {where}
        GROUP BY pickup_zone, dropoff_zone, hour(pickup) HAVING count(*) >= {min_trips}
    """).df()

widths = dict(pickup_zone=160, dropoff_zone=180, hour=70, trips=100, miles=90, fare=150, tip=90)
w = LatticeGridWidget(by_route(), profile=True, histograms=True, height=480, options={"header_height": 90},
                      columns=[{"field": f, "layout": {"width": x}} for f, x in widths.items()],
                      state={"sort": [{"col": "trips", "dir": "desc"}],
                             "filters": {"op": "and", "conditions": [{"col": "fare", "op": "gt", "value": 20}]}})
print(f"{len(w.df):,} routes")
w

14,626 routes


![the grid](https://raw.githubusercontent.com/toclocoinc/lattice-grid-notebooks/main/sql-to-grid-duckdb/grid.png)

*The grid in JupyterLab: a histogram strip in every header, the Statistics tab on the right edge, the sort on trips and the filter on fare. They are still in place after the two `set_data` calls below.*

## 3. Change the query, keep the column state

Same columns, a different query: only trips longer than 5 miles. `w.set_data` swaps in the new result and reloads the rows; the widths, the sort and the `fare` filter stay as they are, and `w.state` is the state the grid holds. (`set_data` expects the same columns: a query that adds or drops columns needs a new widget.)

In [4]:
w.set_data(by_route("miles > 5"))
print(f"{len(w.df):,} routes")
w.state["filters"], w.state["sort"]

1,426 routes


({'op': 'and', 'conditions': [{'col': 'fare', 'op': 'gt', 'value': 20}]},
 [{'col': 'trips', 'dir': 'desc'}])

## 4. A bigger result switches to the large-dataframe mode

Now every route, however rare (`min_trips=1`): about 158,000 rows. Above 100,000 rows the widget stops sending rows up front. The grid asks Python for each window as you scroll, and sort, filter, histograms and the profile are computed over every row. Widths, sort and filter still carry over.

In [5]:
w.set_data(by_route(min_trips=1))
print(f"{len(w.view):,} routes match the filter; large-dataframe mode: {w.windowed}")
w.state["filters"], w.state["sort"]

102,670 routes match the filter; large-dataframe mode: True


({'op': 'and', 'conditions': [{'col': 'fare', 'op': 'gt', 'value': 20}]},
 [{'col': 'trips', 'dir': 'desc'}])

## 5. Back to pandas

`w.view` is the filtered, sorted rows (a lazy handle in the large-dataframe mode); `.head(n)` pulls the first `n` into a DataFrame.

In [6]:
w.view.head(1000).groupby("pickup_zone")[["trips", "fare"]].agg({"trips": "sum", "fare": "mean"}).sort_values("trips", ascending=False).head(5).round(2)

,trips,fare
pickup_zone,,
JFK Airport,55460,67.01
LaGuardia Airport,36762,44.26
Times Sq/Theatre District,7264,52.52
Midtown Center,5949,39.48
Penn Station/Madison Sq West,5812,22.62
